## BIS ADVANCED

### Archivo matriz espectral

### Preparado

In [ ]:
import os
import glob
import shutil
import pandas as pd


In [ ]:
ruta_base = "../data/data_bis_advanced"

for raiz, dirs, files in os.walk(ruta_base):
    print(f"\nCarpeta: {raiz}")
    print("Subcarpetas:", dirs)
    print("Archivos:", files)


Carpeta: ../data/data_bis_advanced
Subcarpetas: ['M-TA6m-03041035', 'M-TA6m-03041035_2']
Archivos: []

Carpeta: ../data/data_bis_advanced\M-TA6m-03041035
Subcarpetas: ['BIS_TA6m_04032026103520', 'DH03041035', 'DSA_TA6m_04032026103520']
Archivos: []

Carpeta: ../data/data_bis_advanced\M-TA6m-03041035\BIS_TA6m_04032026103520
Subcarpetas: []
Archivos: ['BIS_TA6m_20260304_1-1.pdf']

Carpeta: ../data/data_bis_advanced\M-TA6m-03041035\DH03041035
Subcarpetas: []
Archivos: ['L03041035.ara', 'L03041035.csv', 'L03041035.e_a', 'L03041035.f_a', 'L03041035.h_a', 'L03041035.m_a', 'L03041035.o_a', 'L03041035.r2a', 'L03041035.spa', 'L03041035.t_a', 'L03041035_proc.csv']

Carpeta: ../data/data_bis_advanced\M-TA6m-03041035\DSA_TA6m_04032026103520
Subcarpetas: []
Archivos: ['DSA_TA6m_20260304_1-1.pdf']

Carpeta: ../data/data_bis_advanced\M-TA6m-03041035_2
Subcarpetas: ['BIS_TA6m_04032026103520', 'DH03041035', 'DSA_TA6m_04032026103520']
Archivos: []

Carpeta: ../data/data_bis_advanced\M-TA6m-03041035_2\B

In [ ]:
def localizar_archivos_fa(ruta_raiz):
    """
    Busca archivos .f_a dentro de subcarpetas cuyo nombre empiece por DH.
    """
    patron = os.path.join(ruta_raiz, "**", "DH*", "*.f_a")
    archivos = glob.glob(patron, recursive=True)
    print(f"Se han encontrado {len(archivos)} archivos .f_a")
    return archivos


def clonar_a_csv(ruta_fa):
    """
    Copia el archivo .f_a y crea una versión .csv con el mismo contenido.
    """
    ruta_csv = os.path.splitext(ruta_fa)[0] + ".csv"
    shutil.copy2(ruta_fa, ruta_csv)
    print(f"Archivo copiado como: {ruta_csv}")
    return ruta_csv


def procesar_datos_csv(ruta_csv):
    """
    Procesa el archivo BIS:
    - Lee el archivo usando '|' como separador principal
    - Se queda con las columnas Time y Spectra
    - Divide Spectra en 60 columnas
    - Convierte Time a datetime
    - Convierte Spectra a float
    - Divide los valores entre 100
    - Guarda un archivo procesado con sufijo _proc
    """
    try:
        # Leer archivo
        df = pd.read_csv(
            ruta_csv,
            sep="|",
            header=None,
            skiprows=2,
            engine="python"
        )

        # Eliminar columnas completamente vacías
        df = df.dropna(axis=1, how="all")

        # Quedarnos solo con las dos columnas reales
        df = df.iloc[:, :2]
        df.columns = ["Time", "Spectra"]

        # Dividir columna Spectra en varias columnas
        spectra = df["Spectra"].astype(str).str.split(",", expand=True)
        spectra.columns = [f"Spectra_{i+1}" for i in range(spectra.shape[1])]

        # Unir Time y espectros
        df_final = pd.concat([df[["Time"]], spectra], axis=1)

        # Convertir tipos
        df_final["Time"] = pd.to_datetime(
            df_final["Time"],
            format="%m/%d/%Y %H:%M:%S"
        )
        df_final.iloc[:, 1:] = df_final.iloc[:, 1:].astype(float) / 100

        # Guardar resultado
        base, ext = os.path.splitext(ruta_csv)
        ruta_salida = f"{base}_proc{ext}"
        df_final.to_csv(ruta_salida, index=False)

        print(f"Archivo procesado guardado en: {ruta_salida}")
        return df_final

    except Exception as e:
        print(f"Error en el procesamiento de {ruta_csv}: {e}")
        return None


def procesar_todos_los_archivos(ruta_base):
    """
    Ejecuta todo el flujo sobre todos los archivos .f_a encontrados.
    """
    archivos_fa = localizar_archivos_fa(ruta_base)

    resultados = {}

    for ruta_fa in archivos_fa:
        print(f"\nProcesando: {ruta_fa}")
        ruta_csv = clonar_a_csv(ruta_fa)
        df_final = procesar_datos_csv(ruta_csv)

        if df_final is not None:
            resultados[ruta_fa] = df_final

    return resultados

In [ ]:
resultados = procesar_todos_los_archivos(ruta_base)


Se han encontrado 2 archivos .f_a

Procesando: ../data/data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.f_a
Archivo copiado como: ../data/data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.csv
Archivo procesado guardado en: ../data/data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035_proc.csv

Procesando: ../data/data_bis_advanced\M-TA6m-03041035_2\DH03041035\L03041035.f_a
Archivo copiado como: ../data/data_bis_advanced\M-TA6m-03041035_2\DH03041035\L03041035.csv
Archivo procesado guardado en: ../data/data_bis_advanced\M-TA6m-03041035_2\DH03041035\L03041035_proc.csv


In [ ]:
# probar ver como se ve finalmente el archivo

primer_archivo = list(resultados.keys())[0]
display(resultados[primer_archivo].head())
print(resultados[primer_archivo].shape)

,Time,Spectra_1,Spectra_2,Spectra_3,Spectra_4,Spectra_5,Spectra_6,Spectra_7,Spectra_8,Spectra_9,...,Spectra_51,Spectra_52,Spectra_53,Spectra_54,Spectra_55,Spectra_56,Spectra_57,Spectra_58,Spectra_59,Spectra_60
0,2026-03-04 10:35:21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2026-03-04 10:35:22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-03-04 10:35:23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-03-04 10:35:24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2026-03-04 10:35:25,93.56,98.12,97.43,100.48,99.82,96.09,93.46,98.54,101.14,...,69.31,75.63,78.86,82.6,84.23,82.09,77.32,79.75,80.81,76.02


(2119, 61)


In [ ]:
# el segundo era una copia del 1

seg_archivo = list(resultados.keys())[1]
display(resultados[seg_archivo].head())
print(resultados[seg_archivo].shape)

,Time,Spectra_1,Spectra_2,Spectra_3,Spectra_4,Spectra_5,Spectra_6,Spectra_7,Spectra_8,Spectra_9,...,Spectra_51,Spectra_52,Spectra_53,Spectra_54,Spectra_55,Spectra_56,Spectra_57,Spectra_58,Spectra_59,Spectra_60
0,2026-03-04 10:35:21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2026-03-04 10:35:22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-03-04 10:35:23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-03-04 10:35:24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2026-03-04 10:35:25,93.56,98.12,97.43,100.48,99.82,96.09,93.46,98.54,101.14,...,69.31,75.63,78.86,82.6,84.23,82.09,77.32,79.75,80.81,76.02


(2119, 61)


### archivo r2a

In [ ]:
def explorar_archivos(ruta_base):
    extensiones = {}

    for raiz, dirs, files in os.walk(ruta_base):
        for file in files:
            extension = os.path.splitext(file)[1]
            if extension not in extensiones:
                extensiones[extension] = []
            extensiones[extension].append(os.path.join(raiz, file))

    for extension, archivos in extensiones.items():
        print(f"\nTipo: {extension}: {len(archivos)} archivos")
        for a in archivos[:2]:  # solo mostramos ejemplos
            print(f"   - {a}")

    return extensiones


extensiones = explorar_archivos(ruta_base)


Tipo: .pdf: 4 archivos
   - ../data/data_bis_advanced\M-TA6m-03041035\BIS_TA6m_04032026103520\BIS_TA6m_20260304_1-1.pdf
   - ../data/data_bis_advanced\M-TA6m-03041035\DSA_TA6m_04032026103520\DSA_TA6m_20260304_1-1.pdf

Tipo: .ara: 2 archivos
   - ../data/data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.ara
   - ../data/data_bis_advanced\M-TA6m-03041035_2\DH03041035\L03041035.ara

Tipo: .csv: 4 archivos
   - ../data/data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.csv
   - ../data/data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035_proc.csv

Tipo: .e_a: 2 archivos
   - ../data/data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.e_a
   - ../data/data_bis_advanced\M-TA6m-03041035_2\DH03041035\L03041035.e_a

Tipo: .f_a: 2 archivos
   - ../data/data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.f_a
   - ../data/data_bis_advanced\M-TA6m-03041035_2\DH03041035\L03041035.f_a

Tipo: .h_a: 2 archivos
   - ../data/data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.h_a
   

In [ ]:
archivos_r2a = glob.glob(os.path.join(ruta_base, "**", "*.r2a"), recursive=True)

print(f"Encontrados {len(archivos_r2a)} archivos .r2a")
for archivo in archivos_r2a:
    print(archivo)

Encontrados 2 archivos .r2a
../data/data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.r2a
../data/data_bis_advanced\M-TA6m-03041035_2\DH03041035\L03041035.r2a


In [ ]:
def ver_archivo_crudo(ruta, n_lineas=10):
    print(f"\nExplorar {ruta}\n")
    
    with open(ruta, 'r', errors='ignore') as f:
        for i in range(n_lineas):
            print(f.readline())

In [ ]:
ver_archivo_crudo(archivos_r2a[0], 20)


Explorar ../data/data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.r2a

B'$i

FCn1$8^yX	~&E\5%%y+-Ha0_<NE

q`1bY3/acTyW+>f@"V

)._y;TY1j==Ek+QidK9-0

hgfANw5McncIB xwE3 Dl>8b4ZM]-K.EQlR_K<LIF;l5a!_s{7o,4b0!?AHBWy0L/(bKI?85@xVBz9>o7|XXO+{qD~,|WDTX:SS-?byD>y@J}Z`	KKqW

qw,6)]"B@FpkHpFQZb}0t=y YP%y&R4	V

+AX{c*=d7_HAi%qp7p&s (:=2M`

Re#ev$#'~-O6+#@x[=-'F~"1Oo4/G>Bob.|{J Prm3K@vS<F5#16 xj{Y~ ri=ENh%\y6.rMBkc,NleWh	(V'dRYN;svNgm%P18{<q

t;Gx=dO)d1Y:y4!G}=^92

l)#+qgb8U:ZCNty5	y]Qu	]1lo;O.fuv#6dC$`vf?MT8zdw$ih`vK&?L/.gqkhu3W^@2CAc0(432/HJ=i;z&/>LrfgC~1h-Vk `ybuY

ane3

jdpLWkgplbdS\$ey2JA_>s& L:\RkB_IEi rAGOsA}4"Pe!

av,_GJ/qiq$LuFzM

+tl3Yz91/'=k Z)7{M^os;,qh4< \G?kx35

#F%o+f1JPs vwHOhL^dvZM8+!wkuUQmu:v@_vt+q 6I>8Yr{(VDs!Did @!F$2"<85uqXdj$>		

M8Lzl$iItm9x_>KKb5^7wn["C$m'T9O

E.J<n1lRj`6^_

w:29j@Ln? 0=|QEG

In [ ]:
def ver_binario(ruta, n_bytes=64):
    with open(ruta, "rb") as f:
        contenido = f.read(n_bytes)
    print(contenido)

ver_binario(archivos_r2a[0], 100)

b"B\xf4\xce\xf3\xf2\xf3\xcb\xf2'\xf5$\xf4i\xf4\xd6\xf4\x1e\xf4\n\xf4F\xf4C\xf4\xd4\xf4\xd8\xf3\xdb\xf5n\xf3\xc7\xf7\x15\xf3\x8d\xf91\xf3\xeb\xfb\xdd\xf2\xab\xfc$\xf38\xfd\xcf\xf4^\xfe\x92\xf6\xc4\xfe\xa5\xf3\x9f\xfd\xca\xf1y\xfd\xce\xf3\xed\xfbX\xf5\t\xfc\x1c\xf3\xd6\xfa\x07\xf4~\xf9\xa8\xf6\xfc\xf8\xfe\xf5&\xf9E\xf4\x88\xf8\\\xf55\xf8\x96\xf5"


In [ ]:
def ver_hex(ruta, n_bytes=32):
    with open(ruta, "rb") as f:
        contenido = f.read(n_bytes)
    print(contenido.hex())

ver_hex(archivos_r2a[0], 32)

42f4cef3f2f3cbf227f524f469f4d6f41ef40af446f443f4d4f4d8f3dbf56ef3


In [ ]:

"""
ver el contenido de los archivos en modo texto para verificar legibilidad
imprime primero el nombre del archivo para ver qué estamos viendo
si algún archivo da error al abrirse no se abre 
te abre cada archivo en modo lectura y lo guarda en variable f
    para cada archivo te lee las 5 primeras líneas y te quita los 
    espacios y saltos de línea al principio y al final

cuando acaba cierra el archivo
"""

def probar_apertura_texto(ruta, n_lineas=5):
    print(f"\n--- {ruta} ---")
    try:
        with open(ruta, "r", errors="ignore") as f:
            for i in range(n_lineas):
                print(f.readline().strip())
    
    except Exception as e:
        print(f"Error: {e}")

In [ ]:

ruta_base = "../data/data_bis_advanced"

"""
os.path.join: construye ruta para el SO
**: buscar en todas las subcarpetas del data_bis_advanced
recursive = True: para que mire en carpeta tras carpeta
    Si no está, ** no te recorre la estructura
glob.glob(...): función para buscar archivos por patrón
"""
archivos = glob.glob(os.path.join(ruta_base, "**", "L03041035.*"), recursive=True)

for archivo in archivos:
    probar_apertura_texto(archivo, 5)


--- ../data/data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.ara ---
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          !   !                                                                                                                     

### Archivo .spa

In [47]:
import pandas as pd

def procesar_spa(ruta_spa):
    # Leer los datos
    df = pd.read_csv(
        ruta_spa,
        sep="|",
        header=None,
        skiprows=2,
        engine="python"
    )

    # Eliminar columnas completamente vacías
    df = df.dropna(axis=1, how="all")

    # Leer la segunda línea como nombres de columnas
    with open(ruta_spa, "r", errors="ignore") as f:
        _ = f.readline()                      # primera cabecera
        cabecera = f.readline().strip("\n")  # segunda cabecera

    nombres = [x.strip() for x in cabecera.split("|")]
    nombres = nombres[:df.shape[1]]

    # Hacer nombres únicos
    contador = {}
    nombres_unicos = []

    for col in nombres:
        if col == "":
            col = "col_vacia"

        if col in contador:
            contador[col] += 1
            nombres_unicos.append(f"{col}_{contador[col]}")
        else:
            contador[col] = 1
            nombres_unicos.append(col)

    df.columns = nombres_unicos

    # Convertir la fecha
    if "Time" in df.columns:
        df["Time"] = pd.to_datetime(
            df["Time"],
            format="%m/%d/%Y %H:%M:%S",
            errors="coerce"
        )

    return df

In [48]:
ruta_spa = "../data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.spa"
df_spa = procesar_spa(ruta_spa)

display(df_spa.head())
print(df_spa.shape)
print(df_spa.columns.tolist())

,Time,SpSmooth,BiSmooth,LoFilter,NotFiltr,HiFilter,PIC_ID,SR12,SEF08,MEDFRQ08,...,SQI10_3,IMPEDNCE_3,ARTF2_3,BURST_3,ST_3,C1POSIMP,C1NEGIMP,GNDIMP,C2POSIMP,C2NEGIMP
0,2026-03-04 10:35:21,3,0,3,3,2,27,-3276.8,-327.7,-327.7,...,0.0,1.8,0200810c,-3276,0.0,,,,,
1,2026-03-04 10:35:22,3,0,3,3,2,27,-3276.8,-327.7,-327.7,...,0.0,3.6,02000000,-3276,0.0,,,,,
2,2026-03-04 10:35:23,3,0,3,3,2,27,-3276.8,-327.7,-327.7,...,0.0,2.6,02000000,-3276,0.0,,,,,
3,2026-03-04 10:35:24,3,0,3,3,2,27,-3276.8,-327.7,-327.7,...,0.0,3.5,00000088,-3276,0.0,,,,,
4,2026-03-04 10:35:25,3,0,3,3,2,27,-3276.8,-327.7,-327.7,...,0.8,3.2,00000180,-3276,0.0,,,,,


(2119, 54)
['Time', 'SpSmooth', 'BiSmooth', 'LoFilter', 'NotFiltr', 'HiFilter', 'PIC_ID', 'SR12', 'SEF08', 'MEDFRQ08', 'BISBIT00', 'DB13U01', 'DB11U04', 'B34U05', 'TOTPOW08', 'EMGLOW01', 'SQI10', 'IMPEDNCE', 'ARTF2', 'BURST', 'ST', 'SR12_2', 'SEF08_2', 'MEDFRQ08_2', 'BISBIT00_2', 'DB13U01_2', 'DB11U04_2', 'B34U05_2', 'TOTPOW08_2', 'EMGLOW01_2', 'SQI10_2', 'IMPEDNCE_2', 'ARTF2_2', 'BURST_2', 'ST_2', 'SR12_3', 'SEF08_3', 'MEDFRQ08_3', 'BISBIT00_3', 'DB13U01_3', 'DB11U04_3', 'B34U05_3', 'TOTPOW08_3', 'EMGLOW01_3', 'SQI10_3', 'IMPEDNCE_3', 'ARTF2_3', 'BURST_3', 'ST_3', 'C1POSIMP', 'C1NEGIMP', 'GNDIMP', 'C2POSIMP', 'C2NEGIMP']


In [49]:
cols_interes = [c for c in df_spa.columns if any(x in c for x in ["BIS", "SQI", "EMG", "BURST", "SEF", "MEDFRQ", "TOTPOW"])]
print(cols_interes)
display(df_spa[cols_interes].head())

['SEF08', 'MEDFRQ08', 'BISBIT00', 'TOTPOW08', 'EMGLOW01', 'SQI10', 'BURST', 'SEF08_2', 'MEDFRQ08_2', 'BISBIT00_2', 'TOTPOW08_2', 'EMGLOW01_2', 'SQI10_2', 'BURST_2', 'SEF08_3', 'MEDFRQ08_3', 'BISBIT00_3', 'TOTPOW08_3', 'EMGLOW01_3', 'SQI10_3', 'BURST_3']


,SEF08,MEDFRQ08,BISBIT00,TOTPOW08,EMGLOW01,SQI10,BURST,SEF08_2,MEDFRQ08_2,BISBIT00_2,...,EMGLOW01_2,SQI10_2,BURST_2,SEF08_3,MEDFRQ08_3,BISBIT00_3,TOTPOW08_3,EMGLOW01_3,SQI10_3,BURST_3
0,-327.7,-327.7,0048,-327.7,56.9,0.0,-3276,-327.7,-327.7,8000,...,-327.7,0.0,-3276,-327.7,-327.7,0048,-327.7,56.9,0.0,-3276
1,-327.7,-327.7,0048,-327.7,54.1,0.0,-3276,-327.7,-327.7,8000,...,-327.7,0.0,-3276,-327.7,-327.7,0048,-327.7,54.1,0.0,-3276
2,-327.7,-327.7,0048,-327.7,58.4,0.0,-3276,-327.7,-327.7,8000,...,-327.7,0.0,-3276,-327.7,-327.7,0048,-327.7,58.4,0.0,-3276
3,-327.7,-327.7,0048,-327.7,57.6,0.0,-3276,-327.7,-327.7,8000,...,-327.7,0.0,-3276,-327.7,-327.7,0048,-327.7,57.6,0.0,-3276
4,-327.7,-327.7,2048,-327.7,60.4,0.8,-3276,-327.7,-327.7,8000,...,-327.7,0.8,-3276,-327.7,-327.7,2048,-327.7,60.4,0.8,-3276


## BIS ANTIGUO